In [1]:
%load_ext rpy2.ipython

In [18]:
import pandas as pd

import src
import src.load

r_colormap = src.r_colormap
r_out = str(src.OUT)
pd.options.display.float_format = "{:.1f}".format

In [19]:
%%R -i r_colormap -i r_out

suppressMessages(library(tidyverse))
library(ggplot2)
library(ggeffects)
library(here)
library(ggpubr)

options(scipen = 999)

cmap <- setNames(r_colormap$color, r_colormap$channel)

In [4]:
# Load all the Data

channel_df = src.load.channels()
video_df = src.load.videos(filter_period=True, filter_sentences=False)
sentence_df = src.load.sentences(filter_video=False)
popbert_df = src.load.popbert()
comments_df = src.load.comments()


videos = pd.merge(
    channel_df,
    video_df,
    on="channel_id",
    how="inner",
)

sents = sentence_df.merge(popbert_df, on="sentence_id")
sents = sents.groupby("video_id", observed=True).agg(
    n_sents=("video_id", "size"),
    n_elite=("elite", "sum"),
    n_pplcentr=("pplcentr", "sum"),
    avg_elite=("elite", "mean"),
    avg_pplcentr=("pplcentr", "mean"),
)

# Dataset Summary Table

In [5]:
channel_overview = (
    videos.merge(sents, on="video_id")
    .groupby("channel", observed=True)
    .agg(
        ch_videos=("channel", "size"),
        ch_followers=("channel_followers", "first"),
        n_sentences=("n_sents", "sum"),
        n_elite=("n_elite", "sum"),
        n_pplcentr=("n_pplcentr", "sum"),
        avg_likes=("video_likes", "mean"),
        avg_views=("video_views", "mean"),
        avg_duration=("video_duration", "mean"),
        avg_comments=("video_comments", "mean"),
        first_video=("video_uploadtime", "min"),
        latest_video=("video_uploadtime", "max"),
    )
)

In [6]:
channel_overview

,ch_videos,ch_followers,n_sentences,n_elite,n_pplcentr,avg_likes,avg_views,avg_duration,avg_comments,first_video,latest_video
channel,,,,,,,,,,,
AfD TV,1454,250000,142870,17059,3595,3636.7,43377.5,661.0,397.2,2017-12-07,2024-01-19
AfD BT,5220,388000,295280,39749,5985,3859.2,45732.7,436.1,397.4,2017-12-06,2024-01-20
Greens,457,26100,45677,1415,1314,79.6,4512.6,897.6,0.2,2018-01-27,2023-12-13
CDU,632,21900,53112,919,1429,66.5,9282.0,643.2,42.6,2017-12-11,2024-01-19
CSU,145,5170,10357,286,249,35.6,22332.8,442.2,7.5,2017-12-14,2023-10-05
Left,435,29000,45942,2421,1375,257.5,10604.9,869.7,46.1,2017-12-11,2024-01-17
FDP,483,23300,36204,1406,893,0.5,5441.0,641.8,0.7,2018-01-06,2024-01-06
SPD,477,24200,65563,1383,2163,102.1,5168.7,1114.7,25.6,2017-12-07,2024-01-18


In [7]:
# sum of durations

sum_of_seconds = videos.video_duration.sum()
print(f"Total sum of video durations: {round(sum_of_seconds / 60 / 60, 2)} hours")

Total sum of video durations: 1489.35 hours


In [8]:
# number of videos

count_videos = videos.video_id.size
print(f"Total number of valid videos: {count_videos}")

Total number of valid videos: 9628


In [9]:
# number of sentencs

count_sents = channel_overview.n_sentences.sum()
print(f"Total number of valid sentences: {count_sents}")

Total number of valid sentences: 695005


In [10]:
summary_table = channel_overview.drop(["first_video", "latest_video"], axis=1).T
summary_table

channel,AfD TV,AfD BT,Greens,CDU,CSU,Left,FDP,SPD
ch_videos,1454.0,5220.0,457.0,632.0,145.0,435.0,483.0,477.0
ch_followers,250000.0,388000.0,26100.0,21900.0,5170.0,29000.0,23300.0,24200.0
n_sentences,142870.0,295280.0,45677.0,53112.0,10357.0,45942.0,36204.0,65563.0
n_elite,17059.0,39749.0,1415.0,919.0,286.0,2421.0,1406.0,1383.0
n_pplcentr,3595.0,5985.0,1314.0,1429.0,249.0,1375.0,893.0,2163.0
avg_likes,3636.7,3859.2,79.6,66.5,35.6,257.5,0.5,102.1
avg_views,43377.5,45732.7,4512.6,9282.0,22332.8,10604.9,5441.0,5168.7
avg_duration,661.0,436.1,897.6,643.2,442.2,869.7,641.8,1114.7
avg_comments,397.2,397.4,0.2,42.6,7.5,46.1,0.7,25.6


In [11]:
path = src.OUT / "tables/dataset_summary.csv"
summary_table.to_csv(path)

# View Count Violin Plot

In [13]:
df = videos.merge(sents, on="video_id")

In [21]:
%%R -i df -w 1000 -h 600

df_plot <- df %>%
   mutate(
      likes = video_likes + 1,
      views = video_views + 1,
)
like_plot = ggplot(df_plot, aes(x=channel, y=likes, fill=channel)) +
   geom_boxplot(alpha=0.7) +
   scale_y_continuous(trans="log10") +
   scale_color_manual(values=cmap, aesthetics=c("color", "fill")) +
   theme_ggeffects(
      base_family = "serif",
      base_size = 22
   ) +
   theme(
      axis.text.x=element_text(angle=20, hjust=1),
      legend.position = "none"
   ) +
   xlab("Channel") +
   ylab("log10(LikeCount)")

view_plot = ggplot(df_plot, aes(x=channel, y=views, fill=channel)) +
   geom_boxplot(alpha=0.7) +
   scale_y_continuous(trans="log10") +
   scale_color_manual(values=cmap, aesthetics=c("color", "fill")) +
   theme_ggeffects(
      base_family = "serif",
      base_size = 22
   ) +
   theme(
      axis.text.x=element_text(angle=20, hjust=1),
      legend.position = "none"
   ) +
   xlab("Channel") +
   ylab("log10(ViewCount)")

ggarrange(view_plot, like_plot, ncol=2)

ggsave(here(r_out, "/figures/view_count_violin.pdf"))
ggsave(here(r_out, "/figures/view_count_violin.svg"))

Saving 13.9 x 8.33 in image
Saving 13.9 x 8.33 in image


In [23]:
%%R -i df -w 800 -h 1000

df_plot <- df %>%
   mutate(
      likes = video_likes + 1,
      views = video_views + 1,
)
like_plot = ggplot(df_plot, aes(x=channel, y=likes, fill=channel)) +
   geom_boxplot(alpha=0.7) +
   scale_y_continuous(trans="log10") +
   scale_color_manual(values=cmap, aesthetics=c("color", "fill")) +
   theme_ggeffects(
      base_family = "serif",
      base_size = 22
   ) +
   theme(
      axis.text.x=element_text(angle=20, hjust=1),
      legend.position = "none"
   ) +
   xlab("Channel") +
   ylab("log10(LikeCount)")

view_plot = ggplot(df_plot, aes(x=channel, y=views, fill=channel)) +
   geom_boxplot(alpha=0.7) +
   scale_y_continuous(trans="log10") +
   scale_color_manual(values=cmap, aesthetics=c("color", "fill")) +
   theme_ggeffects(
      base_family = "serif",
      base_size = 22
   ) +
   theme(
      axis.text.x=element_text(angle=20, hjust=1),
      legend.position = "none"
   ) +
   xlab("Channel") +
   ylab("log10(ViewCount)")

ggarrange(view_plot, like_plot, ncol=1)

ggsave(here(r_out, "/figures/view_count_violin_vertical.pdf"))
ggsave(here(r_out, "/figures/view_count_violin_vertical.svg"))

Saving 11.1 x 13.9 in image
Saving 11.1 x 13.9 in image
